In [1]:
import os

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv()
llm = ChatOpenRouter(
    model="nvidia/nemotron-3.5-lightning:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0.3,
)

In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Yes, all our cakes including plum and chocolate cakes have eggless versions! Would you like to order one?
Your name is Ravi! Nice to meet you. Would you like to order an eggless cake?


In [ ]:
import csv
from pathlib import Path
from langchain_core.rate_limiters import InMemoryRateLimiter

llm.rate_limiter = InMemoryRateLimiter(requests_per_second=0.32, max_bucket_size=1)

def run_tests(path: str = "test.csv", *, test_chain=chain) -> list[tuple[bool, str]]:
    csv_path = Path(path)
    if not csv_path.is_file():
        csv_path = Path("Day-3") / csv_path
    with csv_path.open(encoding="utf-8-sig", newline="") as source:
        rows = list(csv.DictReader(source))

    results = []
    for row in rows:
        answer = test_chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        results.append((ok, answer))
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])

    passed = sum(ok for ok, _ in results)
    print(f"{passed}/{len(results)} passed ({passed / len(results):.0%})")
    return results

In [4]:
print(f"Original prompt at temperature {llm.temperature}")
baseline = run_tests()

Recorded baseline reuses five saved answers and includes five new answers.
Original prompt at temperature 0.3
PASS | How much is a plum cake? → A plum cake is ₹500. We also have eggless versions available!
PASS | Do you make eggless cakes? → Yes, all our cakes have eggless versions! Just give us 24 hours' notic
PASS | Can I get a cake today evening? → We need 24 hours' notice for all orders, so I can't guarantee a cake f
PASS | Do you deliver to Kakinada? → I'll check with the owner about delivery to Kakinada.
PASS | What is the price of 6 cupcakes? → 6 cupcakes cost Rs 480 (Rs 80 each). Let me know if you'd like to plac
PASS | What is the price of one chocolate cake? → One chocolate cake costs Rs 650. Let me know if you'd like to order on
PASS | Which cakes do you sell? → We sell Plum cake (Rs 500), Chocolate cake (Rs 650), and Cupcakes (Rs 
PASS | Can I order a red velvet cake? → I'll check with the owner about red velvet cake for you! 😊
PASS | How much for one plum cake and four cup

In [5]:
# Add one instruction for this experiment.
revised_system = SYSTEM + "\nFor rush or same-day orders, explicitly say that orders need 24 hours' notice."
revised_prompt = ChatPromptTemplate.from_messages([
    ("system", revised_system),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
revised_chain = revised_prompt | llm | StrOutputParser()

print(f"Revised prompt at temperature {llm.temperature}")
revised = run_tests(test_chain=revised_chain)
change = sum(ok for ok, _ in revised) - sum(ok for ok, _ in baseline)
print(f"Pass-rate change: {100 * change / len(baseline):+.0f} percentage points.")

Revised prompt at temperature 0.3
PASS | How much is a plum cake? → A plum cake is Rs 500. Let me know if you'd like to order one!
PASS | Do you make eggless cakes? → Yes, all our cakes have eggless versions! 🎂 Let me know if you'd like 
PASS | Can I get a cake today evening? → We'd love to bake for you! But all orders need 24 hours' notice, so sa
PASS | Do you deliver to Kakinada? → I'll check with the owner about delivery to Kakinada. All orders need 
PASS | What is the price of 6 cupcakes? → 6 cupcakes cost Rs 480. Would you like to place an order?
PASS | What is the price of one chocolate cake? → One chocolate cake is Rs 650. Happy to help you order one!
PASS | Which cakes do you sell? → We sell plum cake (Rs 500), chocolate cake (Rs 650), and cupcakes (Rs 
PASS | Can I order a red velvet cake? → I'll check with the owner about red velvet cake for you! We have eggle
PASS | How much for one plum cake and four cupcakes? → One plum cake and four cupcakes total ₹820. All our orders nee

In [6]:
hot_chain = revised_prompt | llm.bind(temperature=1.0) | StrOutputParser()

print("Temperature 1.0, run 1")
hot_first = run_tests(test_chain=hot_chain)
print("\nTemperature 1.0, run 2")
hot_second = run_tests(test_chain=hot_chain)

pairs = list(zip(hot_first, hot_second, strict=True))
changed_scores = sum(first[0] != second[0] for first, second in pairs)
changed_replies = sum(first[1].split() != second[1].split() for first, second in pairs)
print(f"{changed_scores}/{len(pairs)} cases changed PASS/FAIL. {changed_replies}/{len(pairs)} replies changed wording.")

Temperature 1.0, run 1
PASS | How much is a plum cake? → A plum cake costs Rs 500. Would you like to order one?
PASS | Do you make eggless cakes? → Yes, all our cakes have eggless versions! Would you like to see the op
PASS | Can I get a cake today evening? → We need 24 hours' notice for all orders, so I can't prepare a cake for
PASS | Do you deliver to Kakinada? → I'll check with the owner about delivery to Kakinada. For any orders, 
PASS | What is the price of 6 cupcakes? → 6 cupcakes cost Rs 480. Each cupcake is Rs 80. Let me know if you'd li
PASS | What is the price of one chocolate cake? → A chocolate cake is Rs 650. Let me know if you'd like to order one!
PASS | Which cakes do you sell? → We sell plum cake (Rs 500), chocolate cake (Rs 650), and cupcakes (Rs 
PASS | Can I order a red velvet cake? → I'll check with the owner about red velvet cake. Note: all orders need
PASS | How much for one plum cake and four cupcakes? → One plum cake and four cupcakes total ₹820. We need 24 hour

In [ ]:
from langchain.agents import create_agent

agent_order = create_agent(model=llm, tools=[], system_prompt=SYSTEM)
result_order = agent_order.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result_order["messages"][-1].content)

Yes, all our cakes have eggless versions! We can make your favorite flavors without eggs. Just let us know your preference when ordering, and we'll need 24 hours' notice.


### Three Sweet Crumbs jobs

- **Plain Python** — Calculate the total from item prices and quantities.
- **One LLM call** — Label a customer message as an order, question, or complaint.
- **Agent** — Clarify an incomplete custom order, check the menu, and price the options before replying.

In [ ]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for a quantity of a menu item."""
    if qty < 1:
        raise ValueError("Quantity must be at least one.")
    return MENU[item.strip().lower()] * qty

def is_on_menu(item: str) -> bool:
    """Check whether the bakery lists the item on its menu."""
    return item.strip().lower() in MENU